# LeetCode 1993 — Operations on Tree bằng Python

Link đề: https://leetcode.com/problems/operations-on-tree/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Tree, Depth-First Search, Breadth-First Search, Design

Skeleton LeetCode:

```python
class LockingTree:

    def __init__(self, parent: List[int]):
        ...

    def lock(self, num: int, user: int) -> bool:
        ...

    def unlock(self, num: int, user: int) -> bool:
        ...

    def upgrade(self, num: int, user: int) -> bool:
        ...
```

Bài **thiết kế cấu trúc dữ liệu** trên cây cho bởi mảng `parent`. Notebook xây một **bộ chạy lệnh** (command runner) để so sánh mọi cài đặt với một bản **ngây thơ** dễ kiểm chứng, rồi đi từ **duyệt toàn bộ cây mỗi lần nâng cấp** → **danh sách con + DFS lặp** → **đếm số con cháu đang khóa** (cắt tỉa DFS) → **Euler tour + bisect**.

## 1. Mục tiêu học

1. Dựng **danh sách con** từ mảng `parent` và duyệt cây **không đệ quy** (stack).
2. Kiểm tra **tổ tiên** bằng cách đi ngược theo `parent` — `O(độ cao)`.
3. Kiểm tra / thao tác trên **con cháu** bằng DFS, bằng **bộ đếm con cháu đang khóa**, hoặc bằng **Euler tour** (đoạn `[tin, tout]`).
4. Viết **command runner** và kiểm thử một lớp thiết kế bằng **đối chiếu ngẫu nhiên** với cài đặt ngây thơ.
5. Nhận ra lỗi kiểm thiếu điều kiện của `upgrade` và lỗi quên cập nhật trạng thái phụ.

## 2. Đề bài & ví dụ

Cho cây `n` đỉnh `0..n−1` qua mảng `parent` (`parent[0] = −1`, gốc là `0`). Thiết kế `LockingTree`:

- `lock(num, user)`: khóa `num` cho `user` nếu `num` **đang mở**; trả `true`/`false`.
- `unlock(num, user)`: mở `num` nếu nó **đang bị chính `user` khóa**; trả `true`/`false`.
- `upgrade(num, user)`: khóa `num` cho `user` **và mở mọi con cháu** của nó (bất kể ai khóa), chỉ khi **cả ba** điều kiện đúng:
  1. `num` đang mở;
  2. `num` có **ít nhất một con cháu** đang bị khóa (bởi bất kỳ ai);
  3. `num` **không có tổ tiên nào** đang bị khóa.

**Ví dụ 1:**
```
["LockingTree","lock","unlock","unlock","lock","upgrade","lock"]
[[[-1,0,0,1,1,2,2]],[2,2],[2,3],[2,2],[4,5],[0,1],[0,1]]
→ [null,true,false,true,true,true,false]
```

**Ràng buộc:** `2 ≤ n ≤ 2000`, `1 ≤ user ≤ 10⁴`, tối đa `2000` lời gọi `lock`/`unlock`/`upgrade`.

In [1]:
import random
import time
from bisect import bisect_left, insort
from typing import List

EX_PARENT = [-1, 0, 0, 1, 1, 2, 2]
EX_OPS = [("lock", 2, 2), ("unlock", 2, 3), ("unlock", 2, 2), ("lock", 4, 5), ("upgrade", 0, 1), ("lock", 0, 1)]
EX_OUT = [True, False, True, True, True, False]

def draw_tree(parent, locked=None):
    children = [[] for _ in parent]
    for v, p in enumerate(parent):
        if p >= 0:
            children[p].append(v)
    lines = []
    stack = [(0, 0)]
    while stack:
        v, d = stack.pop()
        tag = f" [khóa bởi {locked[v]}]" if locked and v in locked else ""
        lines.append("    " + "  " * d + f"{v}{tag}")
        for c in reversed(children[v]):
            stack.append((c, d + 1))
    print("\n".join(lines))

draw_tree(EX_PARENT)

    0
      1
        3
        4
      2
        5
        6


## 3. Kiến thức nền

### 3.1 Từ mảng `parent` sang danh sách con

In [2]:
def build_children(parent):
    children = [[] for _ in parent]
    for v in range(1, len(parent)):
        children[parent[v]].append(v)
    return children

print(build_children(EX_PARENT))

[[1, 2], [3, 4], [5, 6], [], [], [], []]


### 3.2 Đi lên tổ tiên

`v = parent[v]` cho đến `−1`. Chi phí `O(độ cao)`; cây "dây xích" có độ cao `n − 1`.

In [3]:
def ancestors(parent, v):
    out = []
    v = parent[v]
    while v != -1:
        out.append(v); v = parent[v]
    return out

print("tổ tiên của 6:", ancestors(EX_PARENT, 6), "| của 0:", ancestors(EX_PARENT, 0))

tổ tiên của 6: [2, 0] | của 0: []


### 3.3 Duyệt con cháu bằng stack (không đệ quy)

Cây dây xích 2000 đỉnh sẽ vượt giới hạn đệ quy 1000 của Python nếu dùng DFS đệ quy.

In [4]:
def descendants(children, v):
    out, stack = [], list(children[v])
    while stack:
        x = stack.pop(); out.append(x); stack.extend(children[x])
    return out

print("con cháu của 1:", descendants(build_children(EX_PARENT), 1))

con cháu của 1: [4, 3]


### 3.4 Euler tour: con cháu là một đoạn liên tục

Đánh số `tin[v]` khi vào đỉnh, `tout[v]` = `tin` lớn nhất trong cây con. Khi đó `u` là con cháu (hoặc chính) `v` ⇔ `tin[v] ≤ tin[u] ≤ tout[v]`.

In [5]:
def euler(children):
    n = len(children); tin = [0] * n; tout = [0] * n; t = 0
    stack = [(0, False)]
    while stack:
        v, done = stack.pop()
        if done:
            tout[v] = t - 1; continue
        tin[v] = t; t += 1
        stack.append((v, True))
        for c in reversed(children[v]):
            stack.append((c, False))
    return tin, tout

tin, tout = euler(build_children(EX_PARENT))
for v in range(7):
    print(f"  đỉnh {v}: tin={tin[v]}, tout={tout[v]}")

  đỉnh 0: tin=0, tout=6
  đỉnh 1: tin=1, tout=3
  đỉnh 2: tin=4, tout=6
  đỉnh 3: tin=2, tout=2
  đỉnh 4: tin=3, tout=3
  đỉnh 5: tin=5, tout=5
  đỉnh 6: tin=6, tout=6


### 3.5 Command runner cho bài thiết kế

In [6]:
def run(cls, parent, ops):
    obj = cls(parent)
    return [getattr(obj, name)(num, user) for name, num, user in ops]

def run_leetcode(cls, names, args):
    """Chạy theo định dạng LeetCode: names[0] là tên lớp, args[0] là tham số khởi tạo."""
    obj = cls(*args[0]); out = [None]
    for name, a in zip(names[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

## 4. Ý tưởng nền tảng / trực giác

- `lock`/`unlock` chỉ cần **từ điển** `locked[v] = user` — `O(1)`.
- `upgrade` khó hơn: hỏi về **tổ tiên** (đi lên) và **con cháu** (đi xuống).

Ví dụ 1 sau `lock(4, 5)`: đỉnh 0 đang mở ✓, có con cháu 4 đang khóa ✓, không có tổ tiên ✓ ⇒ `upgrade(0, 1)` thành công: khóa 0 cho user 1, mở 4.

## 5. Cách 1 — Brute Force / baseline: duyệt mọi đỉnh mỗi lần nâng cấp

Không dựng danh sách con. Để biết `u` có phải con cháu của `num`: đi lên từ `u` xem có gặp `num` không. `upgrade` duyệt **mọi** đỉnh như thế. Chậm (`O(n · h)` mỗi lần) nhưng **rất dễ thấy đúng** ⇒ dùng làm chuẩn đối chiếu.

In [7]:
class LockingTreeNaive:
    def __init__(self, parent):
        self.parent = parent
        self.locked = {}

    def _is_descendant(self, u, v):           # u là con cháu thật sự của v?
        u = self.parent[u]
        while u != -1:
            if u == v:
                return True
            u = self.parent[u]
        return False

    def lock(self, num, user):
        if num in self.locked:
            return False
        self.locked[num] = user
        return True

    def unlock(self, num, user):
        if self.locked.get(num) != user:
            return False
        del self.locked[num]
        return True

    def upgrade(self, num, user):
        if num in self.locked:
            return False
        if any(self._is_descendant(num, a) for a in self.locked):     # tổ tiên đang khóa
            return False
        desc = [u for u in self.locked if self._is_descendant(u, num)]
        if not desc:
            return False
        for u in desc:
            del self.locked[u]
        self.locked[num] = user
        return True

print(run(LockingTreeNaive, EX_PARENT, EX_OPS), "| kỳ vọng", EX_OUT)

[True, False, True, True, True, False] | kỳ vọng [True, False, True, True, True, False]


**Độ phức tạp:** `upgrade` `O(|locked| · h)`, tối đa `O(n²)` với cây dây xích. Với `2000` lời gọi và `n = 2000` vẫn có thể tới `8·10⁹` bước trong trường hợp xấu — quá chậm.

**Nút thắt:** tìm con cháu bằng cách thử **mọi** đỉnh. Dựng danh sách con để đi xuống trực tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Danh sách con** ⇒ duyệt cây con của `num` trong `O(kích thước cây con)`.
2. **Tổ tiên** ⇒ đi lên `O(h)`.
3. **Bộ đếm** `cnt[v]` = số con cháu (thật sự) đang khóa. `lock`/`unlock` cập nhật mọi tổ tiên (`O(h)`). Khi đó điều kiện 2 kiểm tra `O(1)`; và khi mở con cháu, DFS **chỉ đi vào** cây con có `cnt > 0` hoặc chính nó đang khóa.
4. **Euler tour + danh sách `tin` đang khóa đã sắp** ⇒ "có con cháu khóa không" = có phần tử trong `[tin+1, tout]` — `bisect`.

**Nếu** `num` đang khóa **thì** sai; **nếu** có tổ tiên khóa **thì** sai; **nếu** không có con cháu khóa **thì** sai; **ngược lại** mở hết con cháu và khóa `num`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Danh sách con + DFS lặp, mảng `owner`

In [8]:
class LockingTreeDFS:
    def __init__(self, parent):
        self.parent = parent
        self.children = build_children(parent)
        self.owner = [0] * len(parent)                  # 0 = đang mở (user ≥ 1)

    def lock(self, num, user):
        if self.owner[num]:
            return False
        self.owner[num] = user
        return True

    def unlock(self, num, user):
        if self.owner[num] != user:
            return False
        self.owner[num] = 0
        return True

    def upgrade(self, num, user):
        if self.owner[num]:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        found = False
        stack = list(self.children[num])
        while stack:
            x = stack.pop()
            if self.owner[x]:
                found = True
                self.owner[x] = 0                       # mở luôn trong lúc duyệt
            stack.extend(self.children[x])
        if not found:
            return False
        self.owner[num] = user
        return True

print(run(LockingTreeDFS, EX_PARENT, EX_OPS), "| kỳ vọng", EX_OUT)

[True, False, True, True, True, False] | kỳ vọng [True, False, True, True, True, False]


**Chú ý:** mở con cháu **trong lúc** duyệt là an toàn vì nếu không tìm thấy đỉnh nào khóa thì cũng chẳng có gì bị đổi. Mỗi lời gọi `upgrade` `O(n)`, `lock`/`unlock` `O(1)`.

### 7.2 Cách 3 — Đếm số con cháu đang khóa (DFS cắt tỉa)

In [9]:
class LockingTreeCount:
    def __init__(self, parent):
        self.parent = parent
        self.children = build_children(parent)
        self.owner = [0] * len(parent)
        self.cnt = [0] * len(parent)                    # số con cháu thật sự đang khóa

    def _add_up(self, v, delta):
        v = self.parent[v]
        while v != -1:
            self.cnt[v] += delta
            v = self.parent[v]

    def lock(self, num, user):
        if self.owner[num]:
            return False
        self.owner[num] = user
        self._add_up(num, +1)
        return True

    def unlock(self, num, user):
        if self.owner[num] != user:
            return False
        self.owner[num] = 0
        self._add_up(num, -1)
        return True

    def upgrade(self, num, user):
        if self.owner[num] or self.cnt[num] == 0:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        removed = self.cnt[num]
        stack = [c for c in self.children[num] if self.owner[c] or self.cnt[c]]
        while stack:
            x = stack.pop()
            if self.owner[x]:
                self.owner[x] = 0
            stack.extend(c for c in self.children[x] if self.owner[c] or self.cnt[c])
            self.cnt[x] = 0
        self.cnt[num] = 0
        self._add_up(num, -removed)                     # tổ tiên của num mất 'removed' đỉnh khóa
        self.owner[num] = user
        self._add_up(num, +1)                           # ... và có thêm chính num
        return True

print(run(LockingTreeCount, EX_PARENT, EX_OPS), "| kỳ vọng", EX_OUT)

[True, False, True, True, True, False] | kỳ vọng [True, False, True, True, True, False]


**Độ phức tạp:** `lock`/`unlock` `O(h)`; `upgrade` `O(h + số đỉnh trên các đường tới đỉnh đang khóa)` — chỉ đi vào nhánh có khóa. Điều kiện 2 kiểm tra trong `O(1)`, nên các lời gọi `upgrade` **thất bại** rẻ.

### 7.3 Cách 4 — Euler tour + danh sách `tin` đang khóa (bisect)

In [10]:
class LockingTreeEuler:
    def __init__(self, parent):
        self.parent = parent
        self.tin, self.tout = euler(build_children(parent))
        self.order = [0] * len(parent)                  # order[t] = đỉnh có tin = t
        for v, t in enumerate(self.tin):
            self.order[t] = v
        self.owner = [0] * len(parent)
        self.locked_tins = []                           # các tin đang khóa, đã sắp

    def lock(self, num, user):
        if self.owner[num]:
            return False
        self.owner[num] = user
        insort(self.locked_tins, self.tin[num])
        return True

    def unlock(self, num, user):
        if self.owner[num] != user:
            return False
        self.owner[num] = 0
        self.locked_tins.pop(bisect_left(self.locked_tins, self.tin[num]))
        return True

    def upgrade(self, num, user):
        if self.owner[num]:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        lo = bisect_left(self.locked_tins, self.tin[num] + 1)
        hi = bisect_left(self.locked_tins, self.tout[num] + 1)
        if lo == hi:
            return False                                 # không có con cháu khóa
        for t in self.locked_tins[lo:hi]:
            self.owner[self.order[t]] = 0
        del self.locked_tins[lo:hi]                      # xóa cả đoạn một lần
        self.owner[num] = user
        insort(self.locked_tins, self.tin[num])
        return True

print(run(LockingTreeEuler, EX_PARENT, EX_OPS), "| kỳ vọng", EX_OUT)

[True, False, True, True, True, False] | kỳ vọng [True, False, True, True, True, False]


**Độ phức tạp:** `insort`/`pop` là `O(k)` dịch chuyển trên list (`k` = số đỉnh khóa) nhưng rất nhanh trong thực tế (memmove). `upgrade`: `O(h + log k + số đỉnh được mở + k)`. Mỗi đỉnh bị mở chỉ sau khi đã bị khóa ⇒ tổng số lần mở bị chặn bởi tổng số lần khóa.

## 8. TRACE từng bước

### 8.1 Trace ví dụ 1 với cách 3 (in `owner` và `cnt` sau mỗi lệnh)

In [11]:
def trace(cls, parent, ops):
    obj = cls(parent)
    for name, num, user in ops:
        r = getattr(obj, name)(num, user)
        locked = {v: u for v, u in enumerate(obj.owner) if u}
        extra = f" cnt={obj.cnt}" if hasattr(obj, "cnt") else ""
        print(f"  {name}({num}, {user}) -> {str(r):<5} khóa={locked}{extra}")

trace(LockingTreeCount, EX_PARENT, EX_OPS)

  lock(2, 2) -> True  khóa={2: 2} cnt=[1, 0, 0, 0, 0, 0, 0]
  unlock(2, 3) -> False khóa={2: 2} cnt=[1, 0, 0, 0, 0, 0, 0]
  unlock(2, 2) -> True  khóa={} cnt=[0, 0, 0, 0, 0, 0, 0]
  lock(4, 5) -> True  khóa={4: 5} cnt=[1, 1, 0, 0, 0, 0, 0]
  upgrade(0, 1) -> True  khóa={0: 1} cnt=[0, 0, 0, 0, 0, 0, 0]
  lock(0, 1) -> False khóa={0: 1} cnt=[0, 0, 0, 0, 0, 0, 0]


### 8.2 Trace các điều kiện của `upgrade` trên cây tự dựng (đã đối chiếu bản ngây thơ)

In [12]:
def trace_upgrade(parent, setup, num, user):
    t = LockingTreeDFS(parent)
    for name, a, b in setup:
        getattr(t, name)(a, b)
    locked = {v: u for v, u in enumerate(t.owner) if u}
    draw_tree(parent, locked)
    c1 = t.owner[num] == 0
    anc = ancestors(parent, num)
    c3 = not any(t.owner[a] for a in anc)
    desc = descendants(t.children, num)
    c2 = any(t.owner[d] for d in desc)
    print(f"  upgrade({num}, {user}): (1) đang mở? {c1} | (2) có con cháu khóa? {c2} (con cháu {desc}) | (3) không tổ tiên khóa? {c3} (tổ tiên {anc})")
    r = t.upgrade(num, user)
    naive = LockingTreeNaive(parent)
    for name, a, b in setup:
        getattr(naive, name)(a, b)
    print(f"  => {r} | bản ngây thơ: {naive.upgrade(num, user)}")

P = [-1, 0, 0, 1, 1, 3, 2]
trace_upgrade(P, [("lock", 5, 7), ("lock", 4, 8)], 1, 9)
print()
trace_upgrade(P, [("lock", 5, 7), ("lock", 0, 8)], 3, 9)

    0
      1
        3
          5 [khóa bởi 7]
        4 [khóa bởi 8]
      2
        6
  upgrade(1, 9): (1) đang mở? True | (2) có con cháu khóa? True (con cháu [4, 3, 5]) | (3) không tổ tiên khóa? True (tổ tiên [0])
  => True | bản ngây thơ: True

    0 [khóa bởi 8]
      1
        3
          5 [khóa bởi 7]
        4
      2
        6
  upgrade(3, 9): (1) đang mở? True | (2) có con cháu khóa? True (con cháu [5]) | (3) không tổ tiên khóa? False (tổ tiên [1, 0])
  => False | bản ngây thơ: False


## 9. Code nộp LeetCode

In [13]:
class LockingTree:

    def __init__(self, parent: List[int]):
        self.parent = parent
        self.children = [[] for _ in parent]
        for v in range(1, len(parent)):
            self.children[parent[v]].append(v)
        self.owner = [0] * len(parent)          # 0 = mở; user ≥ 1
        self.cnt = [0] * len(parent)            # số con cháu đang khóa

    def _add_up(self, v: int, delta: int) -> None:
        v = self.parent[v]
        while v != -1:
            self.cnt[v] += delta
            v = self.parent[v]

    def lock(self, num: int, user: int) -> bool:
        if self.owner[num]:
            return False
        self.owner[num] = user
        self._add_up(num, 1)
        return True

    def unlock(self, num: int, user: int) -> bool:
        if self.owner[num] != user:
            return False
        self.owner[num] = 0
        self._add_up(num, -1)
        return True

    def upgrade(self, num: int, user: int) -> bool:
        if self.owner[num] or self.cnt[num] == 0:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        removed = self.cnt[num]
        stack = [num]
        while stack:
            x = stack.pop()
            for c in self.children[x]:
                if self.owner[c] or self.cnt[c]:
                    self.owner[c] = 0
                    stack.append(c)
            self.cnt[x] = 0
        self._add_up(num, 1 - removed)          # mất 'removed' đỉnh khóa dưới num, thêm chính num
        self.owner[num] = user
        return True

out = run_leetcode(LockingTree, ["LockingTree", "lock", "unlock", "unlock", "lock", "upgrade", "lock"],
                   [[[-1, 0, 0, 1, 1, 2, 2]], [2, 2], [2, 3], [2, 2], [4, 5], [0, 1], [0, 1]])
print(out)
assert out == [None] + EX_OUT
print("Ví dụ đề: OK")

[None, True, False, True, True, True, False]
Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**`lock`/`unlock`** làm đúng định nghĩa: `owner[num]` lưu người khóa (0 = mở).

**Bất biến của bộ đếm (cách 3 / code nộp):** `cnt[v]` = số con cháu thật sự của `v` đang khóa.
- `lock(num)` thêm một đỉnh khóa ⇒ mọi tổ tiên `+1` (`_add_up`). `unlock` ngược lại.
- `upgrade(num)`: mọi đỉnh khóa trong cây con thật sự của `num` (đúng `cnt[num]` đỉnh) bị mở ⇒ `cnt` của mọi đỉnh **trong** cây con về 0 (DFS gán `cnt[x] = 0` cho mọi đỉnh đi qua; đỉnh không đi qua có `cnt = 0` và không khóa sẵn), mọi tổ tiên của `num` giảm `removed`; sau đó `num` bị khóa ⇒ tổ tiên `+1`.

**DFS cắt tỉa không bỏ sót:** nếu cây con của `c` chứa đỉnh khóa thì `owner[c] ≠ 0` hoặc `cnt[c] > 0` ⇒ DFS đi vào `c`.

**Ba điều kiện của `upgrade`** được kiểm trực tiếp: `owner[num] == 0`; `cnt[num] > 0` ⇔ có con cháu khóa; đi lên tổ tiên kiểm từng `owner`.

**Cách 4:** `u` là con cháu thật sự của `num` ⇔ `tin[num] < tin[u] ≤ tout[num]`; `locked_tins` luôn là tập `tin` của các đỉnh khóa (đã sắp), nên đoạn `[lo, hi)` là đúng các con cháu đang khóa.

## 11. Phân tích Time / Space Complexity

Gọi `h` là độ cao cây (`≤ n − 1`), `k` là số đỉnh đang khóa.

| Cách | `lock` / `unlock` | `upgrade` | Space |
|---|---|---|---|
| 1. Ngây thơ | `O(1)` | `O(k · h)` | `O(n)` |
| 2. DFS | `O(1)` | `O(h + n)` | `O(n)` |
| 3. Bộ đếm | `O(h)` | `O(h)` nếu thất bại sớm; `O(h + phần cây có khóa)` | `O(n)` |
| 4. Euler + bisect | `O(k)` (insort) | `O(h + k)` | `O(n)` |

In [14]:
def random_parent(n, rnd, shape="random"):
    if shape == "chain":
        return [-1] + list(range(n - 1))
    if shape == "star":
        return [-1] + [0] * (n - 1)
    return [-1] + [rnd.randrange(v) for v in range(1, n)]

def random_ops(n, q, rnd, users=3):
    ops = []
    for _ in range(q):
        name = rnd.choice(["lock", "lock", "unlock", "upgrade"])
        ops.append((name, rnd.randrange(n), rnd.randint(1, users)))
    return ops

rnd = random.Random(1993)
for shape in ["random", "chain", "star"]:
    P = random_parent(2000, rnd, shape); ops = random_ops(2000, 2000, rnd)
    row = []
    for cls in [LockingTreeNaive, LockingTreeDFS, LockingTreeCount, LockingTreeEuler, LockingTree]:
        if cls is LockingTreeNaive and shape == "chain":
            row.append("Naive=(bỏ qua: O(k·h) quá chậm trên dây xích)"); continue
        t0 = time.perf_counter(); run(cls, P, ops); row.append(f"{cls.__name__[11:] or 'Submit'}={(time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"{shape:<6}: " + ", ".join(row))

random: Naive=7ms, DFS=1ms, Count=1ms, Euler=1ms, Submit=1ms
chain : Naive=(bỏ qua: O(k·h) quá chậm trên dây xích), DFS=1ms, Count=33ms, Euler=1ms, Submit=32ms
star  : Naive=21ms, DFS=0ms, Count=0ms, Euler=1ms, Submit=0ms


**Nhận xét từ số đo:** trên cây **dây xích**, cách 3 (và code nộp) chậm hơn cách 2 vì mỗi `lock`/`unlock` phải đi lên `O(h)` tổ tiên, còn cách 2 chỉ tốn `O(1)` cho hai thao tác này. Đổi lại cách 3 làm `upgrade` **thất bại** trong `O(1)` khi không có con cháu khóa. Với ràng buộc của đề (2000 lời gọi, `n ≤ 2000`), cả hai đều rất nhanh — chọn cách nào là đánh đổi giữa chi phí cập nhật và chi phí truy vấn.

## 12. Bộ test

In [15]:
IMPLS = [LockingTreeDFS, LockingTreeCount, LockingTreeEuler, LockingTree]

def check(parent, ops, exp=None, naive=True):
    ref = run(LockingTreeNaive if naive else LockingTreeDFS, parent, ops)
    for cls in IMPLS:
        got = run(cls, parent, ops)
        assert got == ref, (cls.__name__, parent, ops, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

check(EX_PARENT, EX_OPS, EX_OUT)
print("Ví dụ đề: OK")

check([-1, 0], [("upgrade", 0, 1)], [False])                                     # không có con cháu khóa
check([-1, 0], [("lock", 0, 1), ("upgrade", 0, 1)], [True, False])                # num đang khóa
check([-1, 0, 1], [("lock", 0, 1), ("lock", 2, 2), ("upgrade", 1, 3)], [True, True, False])   # tổ tiên khóa
check([-1, 0, 1], [("lock", 2, 1), ("upgrade", 1, 2), ("unlock", 1, 2), ("upgrade", 0, 3)], [True, True, True, False])
check([-1, 0], [("lock", 1, 1), ("unlock", 1, 2), ("unlock", 1, 1), ("unlock", 1, 1)], [True, False, True, False])
check([-1, 0, 0], [("lock", 1, 1), ("lock", 2, 2), ("upgrade", 0, 3), ("lock", 1, 4), ("lock", 0, 5)], [True, True, True, True, False])   # sau upgrade, 1 đã mở nên khóa lại được
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(1500):
    n = rnd.randint(2, 12)
    P = random_parent(n, rnd, rnd.choice(["random", "random", "chain", "star"]))
    check(P, random_ops(n, rnd.randint(1, 40), rnd, users=rnd.randint(1, 3)))
print("1500 chuỗi lệnh ngẫu nhiên trên cây nhỏ: OK")

for _ in range(6):
    n = rnd.randint(200, 2000)
    P = random_parent(n, rnd, rnd.choice(["random", "chain", "star"]))
    check(P, random_ops(n, 2000, rnd, users=5), naive=False)
print("6 chuỗi 2000 lệnh trên cây lớn (đối chiếu chéo, chuẩn là cách 2 đã khớp bản ngây thơ ở trên): OK")

Ví dụ đề: OK
Edge case: OK


1500 chuỗi lệnh ngẫu nhiên trên cây nhỏ: OK


6 chuỗi 2000 lệnh trên cây lớn (đối chiếu chéo, chuẩn là cách 2 đã khớp bản ngây thơ ở trên): OK


In [16]:
# Kiểm bất biến cnt của code nộp sau mỗi lệnh
def check_invariant(t):
    for v in range(len(t.parent)):
        stack = list(t.children[v]); c = 0
        while stack:
            x = stack.pop(); c += bool(t.owner[x]); stack.extend(t.children[x])
        assert t.cnt[v] == c, (v, t.cnt[v], c)

rnd = random.Random(77)
for _ in range(300):
    n = rnd.randint(2, 15); P = random_parent(n, rnd); t = LockingTree(P)
    for name, a, b in random_ops(n, 30, rnd):
        getattr(t, name)(a, b); check_invariant(t)
print("Bất biến cnt[v] = số con cháu đang khóa: luôn đúng trên 300 chuỗi lệnh")

Bất biến cnt[v] = số con cháu đang khóa: luôn đúng trên 300 chuỗi lệnh


## 13. Các lỗi người mới hay gặp

Mỗi lỗi là một lớp con của cách 2 (chỉ sửa một chỗ), chạy trên một chuỗi lệnh ngắn.

### Lỗi 1 — Quên kiểm tra tổ tiên đang khóa

In [17]:
class BugNoAncestor(LockingTreeDFS):
    def upgrade(self, num, user):
        if self.owner[num]:
            return False
        found = False; stack = list(self.children[num])        # SAI: bỏ qua điều kiện (3)
        while stack:
            x = stack.pop()
            if self.owner[x]:
                found = True; self.owner[x] = 0
            stack.extend(self.children[x])
        if not found:
            return False
        self.owner[num] = user
        return True

P, ops = [-1, 0, 1], [("lock", 0, 1), ("lock", 2, 2), ("upgrade", 1, 3)]
print("sai :", run(BugNoAncestor, P, ops), "| đúng:", run(LockingTree, P, ops))

sai : [True, True, True] | đúng: [True, True, False]


**Sửa:** đi lên `parent` và trả `False` nếu gặp đỉnh khóa.

### Lỗi 2 — Quên điều kiện "có ít nhất một con cháu khóa"

In [18]:
class BugNoDescendantCheck(LockingTreeDFS):
    def upgrade(self, num, user):
        if self.owner[num]:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        stack = list(self.children[num])
        while stack:
            x = stack.pop(); self.owner[x] = 0; stack.extend(self.children[x])
        self.owner[num] = user                                  # SAI: nâng cấp cả khi không có gì để mở
        return True

P, ops = [-1, 0], [("upgrade", 0, 1)]
print("sai :", run(BugNoDescendantCheck, P, ops), "| đúng:", run(LockingTree, P, ops))

sai : [True] | đúng: [False]


**Sửa:** ghi nhận `found` và trả `False` nếu không có con cháu khóa (code nộp: `cnt[num] == 0`).

### Lỗi 3 — `unlock` không kiểm tra đúng người khóa

In [19]:
class BugUnlockAnyone(LockingTreeDFS):
    def unlock(self, num, user):
        if not self.owner[num]:                                 # SAI: chỉ kiểm "đang khóa"
            return False
        self.owner[num] = 0
        return True

P, ops = [-1, 0], [("lock", 1, 1), ("unlock", 1, 2)]
print("sai :", run(BugUnlockAnyone, P, ops), "| đúng:", run(LockingTree, P, ops))

sai : [True, True] | đúng: [True, False]


**Sửa:** `if self.owner[num] != user: return False`.

### Lỗi 4 — Mở con cháu nhưng quên khóa chính `num`

In [20]:
class BugForgetLockSelf(LockingTreeDFS):
    def upgrade(self, num, user):
        ok = super().upgrade(num, user)
        if ok:
            self.owner[num] = 0                                 # mô phỏng quên dòng self.owner[num] = user
        return ok

P, ops = [-1, 0], [("lock", 1, 1), ("upgrade", 0, 2), ("lock", 0, 3)]
print("sai :", run(BugForgetLockSelf, P, ops), "| đúng:", run(LockingTree, P, ops))

sai : [True, True, True] | đúng: [True, True, False]


**Vì sao sai:** sau `upgrade` đỉnh 0 phải bị user 2 khóa ⇒ `lock(0, 3)` phải `False`.

### Lỗi 5 — Cách 3: mở con cháu nhưng không cập nhật `cnt`

In [21]:
class BugStaleCount(LockingTreeCount):
    def upgrade(self, num, user):
        if self.owner[num] or self.cnt[num] == 0:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]:
                return False
            v = self.parent[v]
        stack = list(self.children[num])
        while stack:
            x = stack.pop(); self.owner[x] = 0; stack.extend(self.children[x])   # SAI: cnt không đổi
        self.owner[num] = user
        self._add_up(num, +1)
        return True

P, ops = [-1, 0, 1], [("lock", 2, 1), ("upgrade", 1, 2), ("unlock", 1, 2), ("upgrade", 0, 3)]
print("sai :", run(BugStaleCount, P, ops), "| đúng:", run(LockingTree, P, ops))

sai : [True, True, True, True] | đúng: [True, True, True, False]


**Vì sao sai:** sau `upgrade(1)` đỉnh 2 đã mở nhưng `cnt[0]`, `cnt[1]` vẫn đếm nó; sau khi mở đỉnh 1, `cnt[0]` vẫn `> 0` nên `upgrade(0)` tưởng còn con cháu khóa. **Sửa:** đặt `cnt` trong cây con về 0 và trừ `removed` cho tổ tiên. Bài học: **mọi trạng thái phụ phải được cập nhật ở mọi thao tác** — kiểm bằng hàm bất biến như ở mục 12.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài thiết kế: viết **bản ngây thơ đúng** trước, rồi **command runner** đối chiếu ngẫu nhiên.
- Cây từ `parent`: dựng `children`; đi lên bằng `parent`, đi xuống bằng stack.
- "Có con cháu thỏa X?" ⇒ **bộ đếm theo cây con** (cập nhật dọc tổ tiên) hoặc **Euler tour**.
- Cây dây xích `n = 2000` ⇒ tránh đệ quy.
- Dùng `0` làm "mở" được vì `user ≥ 1` — đọc kỹ ràng buộc trước khi chọn giá trị đặc biệt.

## 15. Biến thể của bài

1. **Truy vấn "có tổ tiên khóa?" nhanh hơn `O(h)`:** Euler tour + cây Fenwick đánh dấu đoạn (khóa `v` ⇒ cộng 1 trên `[tin[v], tout[v]]`; hỏi điểm `tin[num]`).
2. **LeetCode 1483 — Kth Ancestor of a Tree Node:** binary lifting trên mảng `parent`.
3. **Khóa đọc/ghi** (shared/exclusive) trên cây thư mục — mô hình "intention lock" trong cơ sở dữ liệu.
4. **`upgrade` chỉ mở các khóa của cùng `user`** (bài tập 5).
5. **Đếm số đỉnh đang khóa trong cây con** — truy vấn `cnt[v]` (bài tập 3).

In [22]:
class Fenwick:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, d):
        i += 1
        while i <= self.n:
            self.t[i] += d; i += i & -i
    def prefix(self, i):
        i += 1; s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s

def locked_ancestor_count_fenwick(parent, locked_nodes, num):
    """Biến thể 1: số tổ tiên (kể cả chính num) đang khóa, bằng cộng đoạn Euler + hỏi điểm."""
    tin, tout = euler(build_children(parent))
    fw = Fenwick(len(parent) + 1)
    for v in locked_nodes:
        fw.add(tin[v], 1); fw.add(tout[v] + 1, -1)
    return fw.prefix(tin[num])

P = [-1, 0, 0, 1, 1, 3, 2]
print("số đỉnh khóa trên đường 0..5 khi khóa {0, 3, 6}:", locked_ancestor_count_fenwick(P, [0, 3, 6], 5), "(kỳ vọng 2: đỉnh 0 và 3)")

số đỉnh khóa trên đường 0..5 khi khóa {0, 3, 6}: 2 (kỳ vọng 2: đỉnh 0 và 3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | `upgrade` | Cập nhật | Khi nào dùng |
|---|---|---|---|---|
| 1. Ngây thơ | thử mọi đỉnh khóa | `O(k h)` | `O(1)` | chuẩn đối chiếu |
| 2. DFS | danh sách con + stack | `O(n)` | `O(1)` | đơn giản, đủ cho ràng buộc |
| 3. Bộ đếm | `cnt[v]`, DFS cắt tỉa | rẻ khi thất bại | `O(h)` | **nộp bài** |
| 4. Euler + bisect | đoạn `[tin, tout]` | `O(h + k)` | `O(k)` | nhiều truy vấn cây con |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Cây `[-1,0,0,1]`, lệnh `lock(3,1)`, `lock(2,2)`, `upgrade(0,3)`, `unlock(3,1)`, `upgrade(1,4)`. Kết quả?

**Bài 2 (tính tay):** Với cây ví dụ `[-1,0,0,1,1,2,2]` (thứ tự con tăng dần), tính `tin`/`tout` của đỉnh 1 và 2.

**Bài 3 (điền TODO):** Thêm phương thức `count_locked_in_subtree(num)` vào `LockingTree` (tính cả chính `num`).

**Bài 4 (tự viết lại):** Viết lại cách 2 nhưng kiểm tra tổ tiên bằng Euler tour + Fenwick (biến thể 1), giữ `lock`/`unlock` cập nhật Fenwick.

**Bài 5 (mini challenge):** Viết `LockingTreeSameUser` trong đó `upgrade(num, user)` chỉ cần có ít nhất một con cháu khóa bởi **chính `user`**, và chỉ mở **các khóa của `user`** (các khóa khác giữ nguyên). Kiểm bằng bản ngây thơ tương ứng.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_locked_in_subtree(tree, num):
    # TODO: tree.cnt[num] + (1 nếu tree.owner[num] else 0)
    pass

# Bài 4 — TODO
class LockingTreeFenwick:
    def __init__(self, parent):
        # TODO: Euler tour, Fenwick cộng đoạn
        pass

# Bài 5 — TODO
class LockingTreeSameUser:
    def __init__(self, parent):
        # TODO
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `lock(3,1)`, `lock(2,2)` đều `True`.
- Mức 2: `upgrade(0,3)` thành công, mở 3 và 2, khóa 0; `unlock(3,1)` ⇒ `False` (3 đã mở).
- Mức 3: `upgrade(1,4)`: tổ tiên 0 đang khóa ⇒ `False`. Kết quả `[True, True, True, False, False]`.

**Bài 2**
- Mức 1: Thứ tự thăm: 0, 1, 3, 4, 2, 5, 6.
- Mức 2: `tin[1] = 1`, cây con {1, 3, 4} ⇒ `tout[1] = 3`.
- Mức 3: `tin[2] = 4`, `tout[2] = 6`.

**Bài 3**
- Mức 1: `cnt` đã là số con cháu thật sự khóa.
- Mức 2: Cộng thêm chính `num`.
- Mức 3: Đối chiếu đếm bằng DFS.

**Bài 4**
- Mức 1: `lock(v)` ⇒ cộng 1 trên `[tin[v], tout[v]]`.
- Mức 2: "Có tổ tiên khóa?" ⇔ `prefix(tin[num]) − (1 nếu num khóa) > 0`.
- Mức 3: `upgrade` mở con cháu phải trừ lại Fenwick cho từng đỉnh được mở.

**Bài 5**
- Mức 1: Duyệt con cháu, chỉ quan tâm `owner == user`.
- Mức 2: Điều kiện (1) và (3) giữ nguyên.
- Mức 3: Bản ngây thơ: lọc `locked` theo user và theo quan hệ con cháu.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
r1 = run(LockingTree, [-1, 0, 0, 1], [("lock", 3, 1), ("lock", 2, 2), ("upgrade", 0, 3), ("unlock", 3, 1), ("upgrade", 1, 4)])
print("Bài 1:", r1); assert r1 == [True, True, True, False, False] == run(LockingTreeNaive, [-1, 0, 0, 1], [("lock", 3, 1), ("lock", 2, 2), ("upgrade", 0, 3), ("unlock", 3, 1), ("upgrade", 1, 4)])

# Bài 2
tin, tout = euler(build_children(EX_PARENT))
print("Bài 2: đỉnh 1:", (tin[1], tout[1]), "| đỉnh 2:", (tin[2], tout[2])); assert (tin[1], tout[1], tin[2], tout[2]) == (1, 3, 4, 6)

# Bài 3
def count_locked_in_subtree_sol(tree, num):
    return tree.cnt[num] + (1 if tree.owner[num] else 0)

rnd = random.Random(3)
for _ in range(300):
    n = rnd.randint(2, 12); P = random_parent(n, rnd); t = LockingTree(P)
    for name, a, b in random_ops(n, 20, rnd):
        getattr(t, name)(a, b)
    for v in range(n):
        assert count_locked_in_subtree_sol(t, v) == sum(1 for u in [v] + descendants(t.children, v) if t.owner[u])
print("Bài 3 OK")

Bài 1: [True, True, True, False, False]
Bài 2: đỉnh 1: (1, 3) | đỉnh 2: (4, 6)
Bài 3 OK


In [25]:
# Bài 4
class LockingTreeFenwickSol(LockingTreeDFS):
    def __init__(self, parent):
        super().__init__(parent)
        self.tin, self.tout = euler(self.children)
        self.fw = Fenwick(len(parent) + 1)
    def _mark(self, v, d):
        self.fw.add(self.tin[v], d); self.fw.add(self.tout[v] + 1, -d)
    def lock(self, num, user):
        ok = super().lock(num, user)
        if ok: self._mark(num, 1)
        return ok
    def unlock(self, num, user):
        ok = super().unlock(num, user)
        if ok: self._mark(num, -1)
        return ok
    def upgrade(self, num, user):
        if self.owner[num] or self.fw.prefix(self.tin[num]) > 0:   # num mở nên prefix chỉ đếm tổ tiên
            return False
        opened = []; stack = list(self.children[num])
        while stack:
            x = stack.pop()
            if self.owner[x]: opened.append(x)
            stack.extend(self.children[x])
        if not opened:
            return False
        for x in opened:
            self.owner[x] = 0; self._mark(x, -1)
        self.owner[num] = user; self._mark(num, 1)
        return True

# Bài 5
class LockingTreeSameUserSol(LockingTreeDFS):
    def upgrade(self, num, user):
        if self.owner[num]:
            return False
        v = self.parent[num]
        while v != -1:
            if self.owner[v]: return False
            v = self.parent[v]
        mine = [x for x in descendants(self.children, num) if self.owner[x] == user]
        if not mine:
            return False
        for x in mine: self.owner[x] = 0
        self.owner[num] = user
        return True

class NaiveSameUser(LockingTreeNaive):
    def upgrade(self, num, user):
        if num in self.locked or any(self._is_descendant(num, a) for a in self.locked):
            return False
        mine = [u for u, w in self.locked.items() if w == user and self._is_descendant(u, num)]
        if not mine:
            return False
        for u in mine: del self.locked[u]
        self.locked[num] = user
        return True

rnd = random.Random(5)
for _ in range(800):
    n = rnd.randint(2, 12); P = random_parent(n, rnd, rnd.choice(["random", "chain", "star"]))
    ops = random_ops(n, rnd.randint(1, 40), rnd, users=rnd.randint(1, 3))
    assert run(LockingTreeFenwickSol, P, ops) == run(LockingTreeNaive, P, ops)
    assert run(LockingTreeSameUserSol, P, ops) == run(NaiveSameUser, P, ops)
print("Bài 4, Bài 5 OK")

Bài 4, Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cây từ `parent` + bộ đếm cây con:**

```python
children = [[] for _ in parent]
for v in range(1, len(parent)): children[parent[v]].append(v)
def add_up(v, d):                      # cập nhật mọi tổ tiên thật sự
    v = parent[v]
    while v != -1: cnt[v] += d; v = parent[v]
```

**Template command runner:**

```python
def run(cls, parent, ops):
    obj = cls(parent)
    return [getattr(obj, name)(*args) for name, *args in ops]
```

**Checklist:**
- [ ] `unlock` kiểm **đúng user**?
- [ ] `upgrade` kiểm đủ **3 điều kiện** (mở, có con cháu khóa, không tổ tiên khóa)?
- [ ] `upgrade` **khóa `num`** sau khi mở con cháu?
- [ ] Trạng thái phụ (`cnt`, Euler list) được cập nhật ở **mọi** thao tác?
- [ ] Không đệ quy trên cây dây xích 2000 đỉnh?
- [ ] Đã đối chiếu ngẫu nhiên với bản ngây thơ?
- [ ] Chữ ký đúng: `LockingTree(parent)`, `lock/unlock/upgrade(self, num: int, user: int) -> bool`?